# 03. Two-Stage Tip Prediction and LLM-vs-Tree Comparison  
# 03. 两阶段小费预测与大模型和树模型对比

**Pipeline position / 主线位置:** clean completed trips -> recorded-tip amount model -> recorded-tip yes/no benchmark.  
**流程位置：** 清洗后的完成订单 -> 记录小费金额模型 -> 是否记录小费的分类对比。

The first part predicts expected recorded tip with two models: one estimates the chance of a positive tip, and the other estimates the amount when a positive tip exists. The second part gives the same 2024 cases to an explainable tree, a LightGBM tree ensemble, and an OpenAI model. `trip_total` is excluded because it already contains the tip and would leak the answer.  
第一部分用两个模型预测记录小费期望值：先估计出现正小费的概率，再估计有小费时的金额。第二部分把相同的 2024 样本交给可解释决策树、LightGBM 树集成和 OpenAI 模型。`trip_total` 已包含小费，因此被排除，避免把答案泄漏给模型。

**Inputs / 输入:** `unified_trips_h3_res9` from Notebook 01.  
**Outputs / 输出:** amount metrics, yes/no metrics, tree rules, model disagreements.

## 0. Install dependencies / 安装依赖

Run the next cell once if the packages are missing.  
如果缺少依赖，运行下一格一次即可。

In [ ]:
# Cell 1 - Install dependencies / 安装依赖
%pip install -q pandas numpy matplotlib scikit-learn pymysql lightgbm holidays requests

## 1. Imports and configuration / 导入包和配置

The default sample is 5,000 valid trips per month, or at most about 180,000 rows across 2022-2024. Reduce `ROWS_PER_MONTH` if the laptop is under pressure.  
默认每个月最多抽取 5,000 条有效行程，2022-2024 合计最多约 18 万行。如果电脑压力较大，可以调低 `ROWS_PER_MONTH`。

In [ ]:
import os
from getpass import getpass
# Cell 2 - Imports and configuration / 导入包和配置

from pathlib import Path
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymysql
import holidays
import lightgbm as lgb

from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    recall_score,
    r2_score,
    roc_auc_score,
)

# Project and output paths / 项目和输出路径 / Portable project configuration / 可移植项目配置
PROJECT_DIR = Path(
    os.getenv("CHICAGO_TNP_PROJECT_DIR", str(Path.cwd()))
).expanduser().resolve()
OUTPUT_DIR = PROJECT_DIR / "notebook_outputs_tip_prediction"
CACHE_DIR = OUTPUT_DIR / "cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# MatrixOne settings / MatrixOne 设置
MO_HOST = os.getenv("MATRIXONE_HOST", "127.0.0.1")
MO_PORT = int(os.getenv("MATRIXONE_PORT", "6001"))
MO_USER = os.getenv("MATRIXONE_USER", "root")
MO_PASSWORD = os.getenv("MATRIXONE_PASSWORD") or getpass("MatrixOne password / MatrixOne 密码: ")
MO_DB = os.getenv("MATRIXONE_DATABASE", "chicago_tnp")
ANALYSIS_TABLE = "unified_trips_h3_res9"

# Lightweight sampling settings / 轻量抽样设置
ROWS_PER_MONTH = 5_000
DATA_START = pd.Timestamp("2022-01-01")
DATA_END_EXCLUSIVE = pd.Timestamp("2025-01-01")
TRAIN_END = pd.Timestamp("2024-01-01")
REUSE_CACHE = True
RANDOM_STATE = 42

# Basic quality filters. These only remove clearly unusable or extreme rows. / 基础质量筛选。这里只去掉明显不可用或极端的记录。
MAX_TIP_SQL = 100
MAX_FARE = 500
MAX_MILES = 100
MAX_SECONDS = 6 * 60 * 60

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("ANALYSIS_TABLE:", ANALYSIS_TABLE)
print("ROWS_PER_MONTH:", ROWS_PER_MONTH)
print("Maximum planned rows:", ROWS_PER_MONTH * 36)
print("OUTPUT_DIR:", OUTPUT_DIR)

## 2. Connect to MatrixOne / 连接 MatrixOne

MatrixOne does the large-table filtering. Jupyter receives only the selected modeling rows. The query helper reconnects and retries once if a long query drops the connection.  
大表筛选由 MatrixOne 完成，Jupyter 只接收选出的建模样本。如果查询过程中连接断开，辅助函数会自动重连并重试一次。

In [ ]:
# Cell 3 - Connect and define SQL helpers / 连接并定义 SQL 工具

def new_connection():
    """Open a new MatrixOne connection. / 创建新的 MatrixOne 连接。"""
    return pymysql.connect(
        host=MO_HOST,
        port=MO_PORT,
        user=MO_USER,
        password=MO_PASSWORD,
        database=MO_DB,
        charset="utf8mb4",
        autocommit=True,
        read_timeout=1800,
        write_timeout=1800,
        local_infile=True,
    )


conn = new_connection()
print("Connected to MatrixOne. / 已连接 MatrixOne。")


def query_df(sql, params=None, retry=True):
    """Run a SQL query and return a pandas DataFrame. / 执行 SQL 查询并返回 pandas DataFrame。"""
    global conn
    try:
        conn.ping(reconnect=True)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return pd.read_sql_query(sql, conn, params=params)
    except pymysql.err.OperationalError:
        if not retry:
            raise
        try:
            conn.close()
        except Exception:
            pass
        conn = new_connection()
        return query_df(sql, params=params, retry=False)


def table_exists(table_name):
    """Check whether a MatrixOne table exists. / 检查 MatrixOne 表是否存在。"""
    sql = """
    SELECT COUNT(*) AS n
    FROM information_schema.tables
    WHERE table_schema = %s AND table_name = %s
    """
    return int(query_df(sql, [MO_DB, table_name]).loc[0, "n"]) > 0

## 3. Validate fields and inspect the full-data tip distribution / 验证字段并查看全量小费分布

This step uses SQL aggregation only. It checks the full table before sampling and confirms that `trip_total` will not enter the feature set.  
这一步只运行 SQL 聚合。在抽样前先检查全量表，并确认 `trip_total` 不会进入模型特征。

In [ ]:
# Cell 4 - Validate table and full-data tip quality / 验证表和全量小费质量

if not table_exists(ANALYSIS_TABLE):
    raise RuntimeError(
        f"Missing table: {ANALYSIS_TABLE}. "
        "Please run the H3 build notebook first. / 请先运行 H3 建表 Notebook。"
    )

schema_df = query_df(f"DESCRIBE {ANALYSIS_TABLE};")
actual_columns = set(schema_df["Field"].astype(str))
required_columns = {
    "trip_start_timestamp", "trip_end_timestamp", "pickup_h3", "dropoff_h3",
    "trip_seconds", "trip_miles", "fare", "tip", "shared_trip_authorized"
}
missing = sorted(required_columns - actual_columns)
if missing:
    raise RuntimeError(f"Missing required columns / 缺少必要字段: {missing}")

full_tip_summary = query_df(f"""
SELECT
    CAST(DATE_FORMAT(trip_start_timestamp, '%Y') AS INT) AS year,
    COUNT(*) AS n,
    SUM(CASE WHEN tip > 0 THEN 1 ELSE 0 END) AS positive_tip_n,
    AVG(CASE WHEN tip > 0 THEN 1.0 ELSE 0.0 END) AS recorded_tip_rate,
    AVG(tip) AS avg_tip_all,
    AVG(CASE WHEN tip > 0 THEN tip ELSE NULL END) AS avg_tip_positive,
    MAX(tip) AS max_tip
FROM {ANALYSIS_TABLE}
WHERE trip_start_timestamp >= '2022-01-01'
  AND trip_start_timestamp < '2025-01-01'
  AND tip IS NOT NULL
  AND tip >= 0
GROUP BY year
ORDER BY year;
""")

display(schema_df)
display(full_tip_summary)

full_tip_summary.to_csv(OUTPUT_DIR / "full_tip_summary_by_year.csv", index=False)

print("Leakage check / 泄漏检查:")
print("trip_total is present in source:", "trip_total" in actual_columns)
print("trip_total will be used as a feature: False")

## 4. Pull a moderate monthly sample / 按月抽取适量样本

Each month contributes at most the same number of rows, so one busy month cannot dominate the sample. Query results are cached locally after the first run.  
每个月最多贡献相同数量的记录，避免某个繁忙月份完全主导样本。第一次查询后会缓存到本地。

This is a lightweight operational sample, not a statistically perfect random sample. The next cells inspect its time and class distribution before modeling.  
这是一个轻量运行样本，不是统计学上完美的随机样本。建模前会检查它的时间分布和小费比例。

In [ ]:
# Cell 5 - Query and cache monthly modeling rows / 查询并缓存按月样本

sample_cache = CACHE_DIR / f"tip_model_sample_{ROWS_PER_MONTH}_per_month.csv.gz"


def month_pairs(start, end_exclusive):
    """Generate consecutive monthly date ranges. / 生成连续的月度日期区间。"""
    starts = pd.date_range(start, end_exclusive, freq="MS", inclusive="left")
    for month_start in starts:
        yield month_start, month_start + pd.offsets.MonthBegin(1)


def fetch_month(month_start, month_end, limit):
    """Load one month of modeling rows from MatrixOne. / 从 MatrixOne 读取一个月的建模样本。"""
    sql = f"""
    SELECT
        trip_start_timestamp,
        trip_end_timestamp,
        pickup_h3,
        dropoff_h3,
        trip_seconds,
        trip_miles,
        fare,
        tip
    FROM {ANALYSIS_TABLE}
    WHERE trip_start_timestamp >= %s
      AND trip_start_timestamp < %s
      AND trip_end_timestamp IS NOT NULL
      AND pickup_h3 IS NOT NULL
      AND dropoff_h3 IS NOT NULL
      AND COALESCE(shared_trip_authorized, 0) = 0
      AND trip_seconds > 0 AND trip_seconds <= %s
      AND trip_miles >= 0 AND trip_miles <= %s
      AND fare >= 0 AND fare <= %s
      AND tip >= 0 AND tip <= %s
    LIMIT {int(limit)};
    """
    return query_df(
        sql,
        [
            month_start.strftime("%Y-%m-%d %H:%M:%S"),
            month_end.strftime("%Y-%m-%d %H:%M:%S"),
            MAX_SECONDS,
            MAX_MILES,
            MAX_FARE,
            MAX_TIP_SQL,
        ],
    )


if REUSE_CACHE and sample_cache.exists():
    print("Loading cached sample / 读取缓存样本:", sample_cache)
    sample_df = pd.read_csv(
        sample_cache,
        compression="gzip",
        parse_dates=["trip_start_timestamp", "trip_end_timestamp"],
    )
else:
    monthly_parts = []
    started = time.time()
    for i, (month_start, month_end) in enumerate(month_pairs(DATA_START, DATA_END_EXCLUSIVE), 1):
        part = fetch_month(month_start, month_end, ROWS_PER_MONTH)
        part["sample_month"] = month_start.strftime("%Y-%m")
        monthly_parts.append(part)
        print(f"{i:02d}/36  {month_start:%Y-%m}: {len(part):,} rows")

    sample_df = pd.concat(monthly_parts, ignore_index=True)
    sample_df["trip_start_timestamp"] = pd.to_datetime(sample_df["trip_start_timestamp"])
    sample_df["trip_end_timestamp"] = pd.to_datetime(sample_df["trip_end_timestamp"])
    sample_df.to_csv(sample_cache, index=False, compression="gzip")
    print(f"Finished in {(time.time() - started) / 60:.1f} minutes.")

print("Sample rows / 样本行数:", f"{len(sample_df):,}")
display(sample_df.head())

## 5. Check sample quality and tip patterns / 检查样本质量和小费规律

Before modeling, confirm that all years and hours are represented and inspect how many trips have a recorded positive tip.  
建模前先确认年份和小时覆盖完整，并检查有多少订单记录了正小费。

In [ ]:
# Cell 6 - Sample diagnostics and tip distribution / 样本诊断和小费分布

sample_df["year"] = sample_df["trip_start_timestamp"].dt.year
sample_df["pickup_hour"] = sample_df["trip_start_timestamp"].dt.hour
sample_df["dropoff_hour"] = sample_df["trip_end_timestamp"].dt.hour
sample_df["has_tip"] = (sample_df["tip"] > 0).astype(int)

sample_quality = (
    sample_df.groupby("year", as_index=False)
    .agg(
        n=("tip", "size"),
        recorded_tip_rate=("has_tip", "mean"),
        avg_tip_all=("tip", "mean"),
        median_tip_all=("tip", "median"),
        avg_tip_positive=("tip", lambda s: s[s > 0].mean()),
    )
)

hour_coverage = sample_df.groupby(["year", "pickup_hour"]).size().unstack(fill_value=0)
positive_tips = sample_df.loc[sample_df["tip"] > 0, "tip"]

display(sample_quality)
display(hour_coverage)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].bar(sample_quality["year"].astype(str), sample_quality["recorded_tip_rate"] * 100)
axes[0].set_title("Recorded positive-tip rate by year")
axes[0].set_ylabel("Percent of sampled trips")

plot_cap = positive_tips.quantile(0.99) if len(positive_tips) else 1
axes[1].hist(positive_tips.clip(upper=plot_cap), bins=40, color="#2E74B5", alpha=0.85)
axes[1].set_title("Positive tip distribution (clipped at sample 99th percentile)")
axes[1].set_xlabel("Recorded tip amount ($)")
axes[1].set_ylabel("Trips")

plt.tight_layout()
plt.show()

sample_quality.to_csv(OUTPUT_DIR / "sample_tip_quality_by_year.csv", index=False)

In [ ]:
# Cell 7 - Time and route pattern exploration / 时间和路线规律探索

pickup_hour_pattern = (
    sample_df.groupby("pickup_hour", as_index=False)
    .agg(
        n=("tip", "size"),
        recorded_tip_rate=("has_tip", "mean"),
        avg_tip=("tip", "mean"),
    )
)
dropoff_hour_pattern = (
    sample_df.groupby("dropoff_hour", as_index=False)
    .agg(
        n=("tip", "size"),
        recorded_tip_rate=("has_tip", "mean"),
        avg_tip=("tip", "mean"),
    )
)

route_summary = (
    sample_df.assign(route=sample_df["pickup_h3"].astype(str) + " -> " + sample_df["dropoff_h3"].astype(str))
    .groupby("route", as_index=False)
    .agg(
        n=("tip", "size"),
        recorded_tip_rate=("has_tip", "mean"),
        avg_tip=("tip", "mean"),
        avg_positive_tip=("tip", lambda s: s[s > 0].mean()),
    )
)
route_summary = route_summary[route_summary["n"] >= 30].sort_values("n", ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
axes[0].plot(pickup_hour_pattern["pickup_hour"], pickup_hour_pattern["recorded_tip_rate"] * 100, marker="o")
axes[0].set_title("Recorded tip rate by pickup hour")
axes[0].set_xlabel("Pickup hour")
axes[0].set_ylabel("Recorded positive-tip rate (%)")

axes[1].plot(dropoff_hour_pattern["dropoff_hour"], dropoff_hour_pattern["recorded_tip_rate"] * 100, marker="o", color="#E67E22")
axes[1].set_title("Recorded tip rate by dropoff hour")
axes[1].set_xlabel("Dropoff hour")

plt.tight_layout()
plt.show()

print("Most frequent routes with at least 30 sampled trips / 样本中最常见的路线:")
display(route_summary.head(20))

pickup_hour_pattern.to_csv(OUTPUT_DIR / "tip_pattern_by_pickup_hour.csv", index=False)
dropoff_hour_pattern.to_csv(OUTPUT_DIR / "tip_pattern_by_dropoff_hour.csv", index=False)
route_summary.to_csv(OUTPUT_DIR / "tip_pattern_by_route.csv", index=False)

## 6. Build leakage-safe features / 构造避免答案泄漏的特征

We split by time first: 2022-2023 for training and 2024 for testing. Category mappings and route frequency are learned from the training period only.  
先按时间切分：2022-2023 训练，2024 测试。地点编码和路线频率只从训练期学习。

In [ ]:
# Cell 8 - Feature engineering and time split / 特征工程和时间切分

df = sample_df.copy()
df["trip_start_timestamp"] = pd.to_datetime(df["trip_start_timestamp"])
df["trip_end_timestamp"] = pd.to_datetime(df["trip_end_timestamp"])

# Calendar features / 日历特征
df["pickup_hour"] = df["trip_start_timestamp"].dt.hour
df["pickup_dow"] = df["trip_start_timestamp"].dt.dayofweek
df["pickup_month"] = df["trip_start_timestamp"].dt.month
df["pickup_is_weekend"] = (df["pickup_dow"] >= 5).astype(int)
df["dropoff_hour"] = df["trip_end_timestamp"].dt.hour
df["dropoff_dow"] = df["trip_end_timestamp"].dt.dayofweek
df["crosses_date"] = (df["trip_start_timestamp"].dt.date != df["trip_end_timestamp"].dt.date).astype(int)

us_holidays = holidays.US(subdiv="IL", years=[2022, 2023, 2024])
df["pickup_is_holiday"] = df["trip_start_timestamp"].dt.date.map(lambda d: int(d in us_holidays))

# Location and route change / 地点与路线变化
df["pickup_h3"] = df["pickup_h3"].astype(str)
df["dropoff_h3"] = df["dropoff_h3"].astype(str)
df["route"] = df["pickup_h3"] + " -> " + df["dropoff_h3"]
df["same_h3"] = (df["pickup_h3"] == df["dropoff_h3"]).astype(int)

# Actual time and completed-trip conditions / 实际时间和完整行程条件
df["timestamp_duration_minutes"] = (
    (df["trip_end_timestamp"] - df["trip_start_timestamp"]).dt.total_seconds() / 60
).clip(lower=0, upper=MAX_SECONDS / 60)
df["speed_mph"] = np.where(
    df["trip_seconds"] > 0,
    df["trip_miles"] / (df["trip_seconds"] / 3600),
    0,
)
df["speed_mph"] = pd.Series(df["speed_mph"], index=df.index).replace([np.inf, -np.inf], np.nan).fillna(0).clip(0, 100)
df["log_fare"] = np.log1p(df["fare"].clip(lower=0))
df["log_miles"] = np.log1p(df["trip_miles"].clip(lower=0))
df["log_seconds"] = np.log1p(df["trip_seconds"].clip(lower=0))
df["has_tip"] = (df["tip"] > 0).astype(int)

train_mask = df["trip_start_timestamp"] < TRAIN_END
train_df = df.loc[train_mask].copy()
test_df = df.loc[~train_mask].copy()

# Remove only the most extreme training-tail tips, then apply the same cap to test. / 仅去掉训练期最极端的小费尾部，再把同一上限应用到测试期。
tip_cap = min(MAX_TIP_SQL, float(train_df["tip"].quantile(0.995)))
train_df = train_df[train_df["tip"] <= tip_cap].copy()
test_df = test_df[test_df["tip"] <= tip_cap].copy()

# Fit categorical mappings on training data only. / 地点编码只在训练数据中拟合。
categorical_source_cols = ["pickup_h3", "dropoff_h3"]
categorical_feature_cols = []
for col in categorical_source_cols:
    mapping = {value: i for i, value in enumerate(train_df[col].drop_duplicates())}
    code_col = col + "_code"
    train_df[code_col] = train_df[col].map(mapping).fillna(-1).astype("int32")
    test_df[code_col] = test_df[col].map(mapping).fillna(-1).astype("int32")
    categorical_feature_cols.append(code_col)

# Route frequency uses training counts only; unseen 2024 routes receive zero. / 路线频率只使用训练期计数；2024 新路线记为 0。
route_counts = train_df["route"].value_counts()
train_df["route_train_frequency"] = train_df["route"].map(route_counts).fillna(0).astype(float)
test_df["route_train_frequency"] = test_df["route"].map(route_counts).fillna(0).astype(float)
train_df["log_route_frequency"] = np.log1p(train_df["route_train_frequency"])
test_df["log_route_frequency"] = np.log1p(test_df["route_train_frequency"])

print("Training rows / 训练行数:", f"{len(train_df):,}")
print("Test rows / 测试行数:", f"{len(test_df):,}")
print("Tip cap learned from training / 训练期小费上限:", round(tip_cap, 2))
print("Train recorded-tip rate:", round(train_df["has_tip"].mean(), 4))
print("Test recorded-tip rate:", round(test_df["has_tip"].mean(), 4))

if train_df["has_tip"].nunique() < 2 or test_df["has_tip"].nunique() < 2:
    raise RuntimeError(
        "The sample does not contain both zero-tip and positive-tip trips. "
        "Increase ROWS_PER_MONTH. / 样本没有同时覆盖零小费和正小费，请提高 ROWS_PER_MONTH。"
    )
if int((train_df["tip"] > 0).sum()) < 500:
    raise RuntimeError(
        "Too few positive-tip training rows. Increase ROWS_PER_MONTH. "
        "/ 正小费训练样本过少，请提高 ROWS_PER_MONTH。"
    )

## 7. Train two-stage models / 训练两阶段模型

For each feature set, the classifier estimates the probability of a recorded tip. The regressor estimates the amount conditional on a positive tip. Their product is the expected recorded tip.  
对于每组特征，分类模型估计“有记录到小费”的概率；回归模型估计“给小费时的金额”；两者相乘得到预计小费。

The three feature sets answer different questions:  
三组特征回答不同问题：

- **Pickup-time information:** can we estimate the tip near trip start, assuming destination is known?  
  **出发时信息：**假设目的地已知，刚上车时能不能估计小费？
- **Time and location change:** how much do actual arrival time and duration add?  
  **时间地点变化：**真实到达时间和时长增加了多少信息？
- **Completed trip:** how much do fare, mileage, seconds, and speed add after completion?  
  **完整行程：**行程结束后，价格、里程、时长和速度又增加多少信息？

In [ ]:
# Cell 9 - Define feature groups and evaluation helpers / 定义特征组和评估函数

pickup_time_features = [
    "pickup_hour", "pickup_dow", "pickup_month", "pickup_is_weekend", "pickup_is_holiday",
    "pickup_h3_code", "dropoff_h3_code", "same_h3", "log_route_frequency",
]

time_location_change_features = pickup_time_features + [
    "dropoff_hour", "dropoff_dow", "crosses_date", "timestamp_duration_minutes",
]

completed_trip_features = time_location_change_features + [
    "log_fare", "log_miles", "log_seconds", "speed_mph",
]

feature_groups = {
    "Pickup-time info": pickup_time_features,
    "Time + location change": time_location_change_features,
    "Completed trip": completed_trip_features,
}


def safe_roc_auc(y_true, probability):
    """Calculate ROC-AUC when both classes are present. / 在正负类别均存在时计算 ROC-AUC。"""
    return roc_auc_score(y_true, probability) if len(np.unique(y_true)) == 2 else np.nan


def regression_metrics(y_true, y_pred):
    """Calculate regression evaluation metrics. / 计算回归评价指标。"""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
    }


def fit_two_stage(train_data, test_data, features, model_name):
    """Train the tip-probability and positive-tip amount models. / 训练小费概率模型与正小费金额模型。"""
    cat_features = [c for c in categorical_feature_cols if c in features]
    X_train = train_data[features].copy()
    X_test = test_data[features].copy()
    y_train_has_tip = train_data["has_tip"].astype(int)
    y_test_has_tip = test_data["has_tip"].astype(int)

    classifier = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
    classifier.fit(X_train, y_train_has_tip, categorical_feature=cat_features)
    tip_probability = classifier.predict_proba(X_test)[:, 1]

    positive_train = train_data["tip"] > 0
    regressor = lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=350,
        learning_rate=0.05,
        num_leaves=31,
        min_child_samples=50,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbosity=-1,
    )
    regressor.fit(
        train_data.loc[positive_train, features],
        np.log1p(train_data.loc[positive_train, "tip"]),
        categorical_feature=cat_features,
    )
    positive_tip_prediction = np.expm1(regressor.predict(X_test))
    positive_tip_prediction = np.clip(positive_tip_prediction, 0, tip_cap)
    expected_tip_prediction = tip_probability * positive_tip_prediction

    predicted_has_tip = (tip_probability >= 0.5).astype(int)
    amount_scores = regression_metrics(test_data["tip"], expected_tip_prediction)
    positive_test = test_data["tip"] > 0
    positive_amount_mae = mean_absolute_error(
        test_data.loc[positive_test, "tip"],
        positive_tip_prediction[positive_test.to_numpy()],
    )

    metrics = {
        "model": model_name,
        "ROC_AUC": safe_roc_auc(y_test_has_tip, tip_probability),
        "PR_AUC": average_precision_score(y_test_has_tip, tip_probability),
        "Brier": brier_score_loss(y_test_has_tip, tip_probability),
        "Precision_at_0.5": precision_score(y_test_has_tip, predicted_has_tip, zero_division=0),
        "Recall_at_0.5": recall_score(y_test_has_tip, predicted_has_tip, zero_division=0),
        "F1_at_0.5": f1_score(y_test_has_tip, predicted_has_tip, zero_division=0),
        "Expected_tip_MAE": amount_scores["MAE"],
        "Expected_tip_RMSE": amount_scores["RMSE"],
        "Expected_tip_R2": amount_scores["R2"],
        "Positive_tip_amount_MAE": positive_amount_mae,
    }
    predictions = pd.DataFrame({
        "tip_probability": tip_probability,
        "positive_tip_prediction": positive_tip_prediction,
        "expected_tip_prediction": expected_tip_prediction,
    }, index=test_data.index)
    return classifier, regressor, metrics, predictions

In [ ]:
# Cell 10 - Train and compare feature groups / 训练并比较不同特征组

trained_models = {}
prediction_sets = {}
metrics_rows = []

# Simple amount baseline: every 2024 trip receives the 2022-2023 mean recorded tip. / 简单金额基线：所有 2024 行程都预测成 2022-2023 平均小费。
baseline_prediction = np.full(len(test_df), train_df["tip"].mean())
baseline_scores = regression_metrics(test_df["tip"], baseline_prediction)
metrics_rows.append({
    "model": "Training mean baseline",
    "ROC_AUC": np.nan,
    "PR_AUC": np.nan,
    "Brier": np.nan,
    "Precision_at_0.5": np.nan,
    "Recall_at_0.5": np.nan,
    "F1_at_0.5": np.nan,
    "Expected_tip_MAE": baseline_scores["MAE"],
    "Expected_tip_RMSE": baseline_scores["RMSE"],
    "Expected_tip_R2": baseline_scores["R2"],
    "Positive_tip_amount_MAE": np.nan,
})

for model_name, features in feature_groups.items():
    print("Training / 正在训练:", model_name)
    classifier, regressor, metrics, predictions = fit_two_stage(
        train_df, test_df, features, model_name
    )
    trained_models[model_name] = {
        "classifier": classifier,
        "regressor": regressor,
        "features": features,
    }
    prediction_sets[model_name] = predictions
    metrics_rows.append(metrics)

model_metrics = pd.DataFrame(metrics_rows)
display(model_metrics)

model_metrics.to_csv(OUTPUT_DIR / "tip_prediction_model_metrics.csv", index=False)

## 8. Read the model comparison / 解读模型对比

- Lower `Expected_tip_MAE` means the expected dollar amount is closer to the recorded tip.  
  `Expected_tip_MAE` 越低，预计小费金额越接近数据中记录的小费。
- Higher `ROC_AUC` and `PR_AUC` mean the model ranks likely tippers better.  
  `ROC_AUC` 和 `PR_AUC` 越高，模型越能把更可能给小费的订单排在前面。
- The improvement from pickup-time to time-location-change shows the value of actual arrival information.  
  从“出发时信息”到“时间地点变化”的提升，代表真实到达信息的价值。
- The improvement from time-location-change to completed-trip shows the added value of fare, miles, seconds, and speed.  
  从“时间地点变化”到“完整行程”的提升，代表价格、里程、时长和速度的额外价值。

In [ ]:
# Cell 11 - Plot model comparison and predicted-vs-actual deciles / 模型对比与预测分组检查

plot_metrics = model_metrics.sort_values("Expected_tip_MAE")
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

axes[0].barh(plot_metrics["model"], plot_metrics["Expected_tip_MAE"], color="#2E74B5")
axes[0].invert_yaxis()
axes[0].set_title("Expected tip MAE by model")
axes[0].set_xlabel("MAE ($, lower is better)")

classification_rows = model_metrics[model_metrics["ROC_AUC"].notna()].copy()
axes[1].barh(classification_rows["model"], classification_rows["ROC_AUC"], color="#E67E22")
axes[1].invert_yaxis()
axes[1].set_xlim(0.5, 1.0)
axes[1].set_title("Recorded-tip classification ROC-AUC")
axes[1].set_xlabel("ROC-AUC (higher is better)")

plt.tight_layout()
plt.show()

best_model_name = model_metrics.loc[
    model_metrics["Expected_tip_MAE"].idxmin(), "model"
]
if best_model_name == "Training mean baseline":
    best_model_name = min(
        feature_groups,
        key=lambda name: float(model_metrics.loc[model_metrics["model"] == name, "Expected_tip_MAE"].iloc[0]),
    )

best_predictions = prediction_sets[best_model_name].copy()
evaluation_df = test_df[[
    "trip_start_timestamp", "pickup_h3", "dropoff_h3", "tip", "has_tip"
]].copy()
evaluation_df = evaluation_df.join(best_predictions)

evaluation_df["prediction_decile"] = pd.qcut(
    evaluation_df["expected_tip_prediction"],
    q=10,
    labels=False,
    duplicates="drop",
) + 1

decile_check = (
    evaluation_df.groupby("prediction_decile", as_index=False)
    .agg(
        n=("tip", "size"),
        avg_actual_tip=("tip", "mean"),
        avg_predicted_tip=("expected_tip_prediction", "mean"),
        actual_recorded_tip_rate=("has_tip", "mean"),
        avg_predicted_tip_probability=("tip_probability", "mean"),
    )
)

display(decile_check)

fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.plot(decile_check["prediction_decile"], decile_check["avg_actual_tip"], marker="o", label="Actual recorded tip")
ax.plot(decile_check["prediction_decile"], decile_check["avg_predicted_tip"], marker="o", label="Predicted expected tip")
ax.set_title(f"Predicted vs actual tip by prediction decile - {best_model_name}")
ax.set_xlabel("Prediction decile (10 = highest predicted tip)")
ax.set_ylabel("Average recorded tip ($)")
ax.legend()
plt.tight_layout()
plt.show()

decile_check.to_csv(OUTPUT_DIR / "tip_prediction_decile_check.csv", index=False)

## 9. Which features matter most? / 哪些特征最重要？

Feature importance shows what helped prediction, not what caused the tip. A high-importance location or hour is an association, not a causal claim.  
特征重要性只说明什么信息对预测有帮助，不代表它造成了小费变化。地点或小时很重要，只能说明存在关联，不能直接说存在因果。

In [ ]:
# Cell 12 - Feature importance for the completed-trip model / 完整行程模型的特征重要性

final_name = "Completed trip"
final_bundle = trained_models[final_name]
final_features = final_bundle["features"]

importance_df = pd.DataFrame({
    "feature": final_features,
    "classifier_importance": final_bundle["classifier"].feature_importances_.astype(float),
    "positive_amount_importance": final_bundle["regressor"].feature_importances_.astype(float),
})

for col in ["classifier_importance", "positive_amount_importance"]:
    total = importance_df[col].sum()
    importance_df[col + "_share"] = importance_df[col] / total if total else 0

importance_df["combined_importance_share"] = (
    importance_df["classifier_importance_share"]
    + importance_df["positive_amount_importance_share"]
) / 2
importance_df = importance_df.sort_values("combined_importance_share", ascending=False)

display(importance_df.head(20))

top_imp = importance_df.head(20).sort_values("combined_importance_share")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_imp["feature"], top_imp["combined_importance_share"], color="#2E74B5")
ax.set_title("Top feature importance - completed-trip two-stage model")
ax.set_xlabel("Combined importance share")
plt.tight_layout()
plt.show()

importance_df.to_csv(OUTPUT_DIR / "tip_prediction_feature_importance.csv", index=False)

## 10. Export predictions and final checks / 导出预测和最终检查

The export contains 2024 test predictions only. It does not contain `trip_total`.  
导出文件只包含 2024 测试期预测，而且不包含 `trip_total`。

In [ ]:
# Cell 13 - Export results and print a compact conclusion / 导出结果并打印简短结论

evaluation_export = evaluation_df.copy()
evaluation_export.to_csv(
    OUTPUT_DIR / "tip_prediction_2024_test_predictions.csv.gz",
    index=False,
    compression="gzip",
)

best_row = model_metrics.loc[model_metrics["Expected_tip_MAE"].idxmin()]
pickup_row = model_metrics.loc[model_metrics["model"] == "Pickup-time info"].iloc[0]
change_row = model_metrics.loc[model_metrics["model"] == "Time + location change"].iloc[0]
completed_row = model_metrics.loc[model_metrics["model"] == "Completed trip"].iloc[0]

print("=" * 90)
print("TIP PREDICTION SUMMARY / 小费预测总结")
print("=" * 90)
print(f"Best expected-tip model: {best_row['model']}")
print(f"Best expected-tip MAE: ${best_row['Expected_tip_MAE']:.3f}")
print(f"Pickup-time MAE: ${pickup_row['Expected_tip_MAE']:.3f}")
print(f"Time + location change MAE: ${change_row['Expected_tip_MAE']:.3f}")
print(f"Completed-trip MAE: ${completed_row['Expected_tip_MAE']:.3f}")
print(f"Completed-trip ROC-AUC: {completed_row['ROC_AUC']:.4f}")
print()
print("Interpretation / 解释:")
print("1. Compare pickup-time vs time-location-change to measure the value of actual arrival time.")
print("   对比出发时模型和时间地点变化模型，判断真实到达时间增加了多少价值。")
print("2. Compare time-location-change vs completed-trip to measure the value of fare, distance, and duration.")
print("   对比时间地点变化模型和完整行程模型，判断价格、距离和时长增加了多少价值。")
print("3. This predicts recorded tips, not necessarily all cash tips.")
print("   预测的是数据中记录到的小费，不一定包含全部现金小费。")
print("4. Feature importance is predictive association, not causality.")
print("   特征重要性表示预测关联，不代表因果。")
print()
print("Outputs / 输出目录:", OUTPUT_DIR)

## 11. Results summary / 结果总结

The experiment establishes four results. First, pickup-time and location features reduce expected-tip MAE from the training-mean baseline of $1.576 to about $1.415. Second, actual dropoff time and location change add little additional expected-tip accuracy in this sample. Third, completed-trip variables improve classification and positive-tip amount estimation, but their expected-tip MAE remains close to the pickup-time model. Fourth, the prediction-decile table rises steadily from low to high groups, showing that the model ranks low- and high-tip trips meaningfully even though individual-trip error remains substantial.  
本实验得到四项结果。第一，上车时间和地点特征将记录小费期望值的 MAE 从训练均值基线的 1.576 美元降到约 1.415 美元。第二，在该样本中，真实下车时间和地点变化对记录小费期望值的额外提升很小。第三，完整行程变量改善了是否记录小费的分类和正小费金额估计，但记录小费期望值的 MAE 仍与上车时间模型接近。第四，预测十分位表从低组到高组稳定上升，说明模型能够有效排序低小费与高小费行程，但单笔行程误差仍然明显。

The results describe predictive associations in completed-trip records. They do not measure unrecorded cash tips and do not establish causal effects.  
这些结果描述的是完成订单记录中的预测关联，不包含未记录现金小费，也不构成因果结论。

## 12. Recorded-Tip Classification: LLM vs Tree  
## 12. 记录小费分类：大模型与树模型  
# 12. 记录小费分类：大模型与树模型对比

This section evaluates whether a positive tip is recorded. LightGBM provides the tree-ensemble accuracy benchmark, a depth-limited decision tree exposes readable decision rules, and the OpenAI model classifies the same answer-free trip summaries.  
本节评估一笔行程是否记录正小费。LightGBM 提供树集成准确率基准，限制深度的决策树展示可读判断规则，OpenAI 模型则对相同且不含答案的行程摘要进行分类。

## 12.1 Experiment design / 实验设计

- **Tree ensemble**: reuse the earlier `Completed trip` LightGBM classifier. / 复用前面的完整行程 LightGBM 分类器。  
- **Explainable tree**: train one depth-4 decision tree and print every split rule. / 训练一棵深度为 4 的决策树并打印全部分裂规则。  
- **LLM**: send human-readable trip attributes and require `yes` or `no`. / 把可读的行程属性发给大模型，强制回答 `yes` 或 `no`。  
- **Evaluation**: accuracy, balanced accuracy, precision, recall, F1, and confusion matrix. / 比较准确率、平衡准确率、精确率、召回率、F1 和混淆矩阵。  

The default LLM sample is intentionally moderate to control cost. It preserves the natural positive-tip rate through stratified sampling.  
默认的大模型测试样本量有意控制在适中水平，并通过分层抽样保持原测试集的正小费比例，以控制费用并保证样本具有代表性。

In [ ]:
# Cell 14 - Cell 1 - Imports and experiment settings / 导入包和实验设置

from pathlib import Path
from getpass import getpass
import json
import os
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree

# Cost and reproducibility settings. / 费用与可复现设置。
LLM_MODEL = "gpt-5.4-nano"
LLM_EVAL_N = 200
LLM_BATCH_SIZE = 10
LLM_MAX_RETRIES = 3

LLM_OUTPUT_DIR = OUTPUT_DIR / "llm_tip_presence_comparison"
LLM_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
safe_model_name = LLM_MODEL.replace("/", "_")
LLM_CACHE_PATH = LLM_OUTPUT_DIR / f"llm_tip_presence_{safe_model_name}_{LLM_EVAL_N}.jsonl"

print("LLM model / 大模型:", LLM_MODEL)
print("Planned evaluation rows / 计划测试行数:", LLM_EVAL_N)
print("This version requires a real API call in Cell 6. / 本版本要求在 Cell 6 完成真实 API 调用。")
print("Cache / 缓存:", LLM_CACHE_PATH)

In [ ]:
import os
from getpass import getpass
# Experiment 2 - Cell 2 - Enter the API key safely / 安全输入 API key

# The key stays only in this kernel and is never written to the notebook or output files. / API key 只保存在当前 kernel 内存中，不会写入 Notebook 或输出文件。
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "").strip()

if not OPENAI_API_KEY:
    OPENAI_API_KEY = getpass("Paste OPENAI_API_KEY (input is hidden) / 输入 API key（不会显示）: ").strip()

if not OPENAI_API_KEY:
    raise EnvironmentError("OPENAI_API_KEY is required. / 必须输入 OPENAI_API_KEY。")

# Validate both the key and access to the selected model before running 200 cases. / 在执行 200 条样本前，先验证 key 以及当前账号是否能访问所选模型。
verify_response = requests.get(
    f"https://api.openai.com/v1/models/{LLM_MODEL}",
    headers={"Authorization": f"Bearer {OPENAI_API_KEY}"},
    timeout=60,
)
if verify_response.status_code >= 400:
    raise RuntimeError(
        f"OpenAI connection check failed ({verify_response.status_code}): "
        f"{verify_response.text[:800]} / OpenAI 连接验证失败。"
    )

print("API key ready / API key 已准备:", bool(OPENAI_API_KEY))
print("OpenAI connection and model access verified / OpenAI 连接及模型权限验证成功:", LLM_MODEL)
print("The key value is intentionally not printed. / 不会打印 key 内容。")

## 12.2 Use the same 2024 rows for every model / 所有模型使用相同的 2024 样本

The comparison must be apples-to-apples. We first draw one fixed, stratified subset from `test_df`, then ask every method to classify those exact rows.  
为了公平比较，我们先从 `test_df` 固定抽取一份保持原正小费比例的样本，然后让所有方法判断完全相同的行程。

In [ ]:
# Cell 16 - Cell 3 - Build one shared evaluation set / 构造统一测试集

required_objects = ["train_df", "test_df", "trained_models", "final_features"]
missing_objects = [name for name in required_objects if name not in globals()]
if missing_objects:
    raise RuntimeError(
        f"Run the original notebook through Cell 13 first. Missing: {missing_objects} / "
        f"请先运行原 Notebook 到 Cell 13。缺少变量: {missing_objects}"
    )

eval_n = min(LLM_EVAL_N, len(test_df))
if eval_n < 20:
    raise RuntimeError("The 2024 test set is too small. / 2024 测试集太小。")

# Stratified sampling keeps the original 2024 positive-tip rate. / 分层抽样保留 2024 原有正小费比例。
_, llm_eval_df = train_test_split(
    test_df,
    test_size=eval_n,
    stratify=test_df["has_tip"],
    random_state=RANDOM_STATE,
)
llm_eval_df = llm_eval_df.sort_values("trip_start_timestamp").copy()
llm_eval_df["case_id"] = [f"tip_case_{i:04d}" for i in range(1, len(llm_eval_df) + 1)]

X_shared = llm_eval_df[final_features].copy()
y_shared = llm_eval_df["has_tip"].astype(int).to_numpy()

# Existing LightGBM tree ensemble. / 原有 LightGBM 树集成分类器。
tree_ensemble = trained_models["Completed trip"]["classifier"]
tree_probability = tree_ensemble.predict_proba(X_shared)[:, 1]
tree_prediction = (tree_probability >= 0.5).astype(int)

# Majority-class baseline. / 多数类基准。
majority_class = int(train_df["has_tip"].mode().iloc[0])
majority_prediction = np.full(len(llm_eval_df), majority_class, dtype=int)

print("Shared evaluation rows / 统一测试行数:", len(llm_eval_df))
print("Actual positive-tip rate / 实际正小费比例:", round(y_shared.mean(), 4))
print("Majority class / 多数类别:", "yes" if majority_class else "no")
display(llm_eval_df[[
    "case_id", "trip_start_timestamp", "pickup_h3", "dropoff_h3",
    "fare", "trip_miles", "trip_seconds", "has_tip"
]].head())

## 12.3 Show how a tree makes decisions / 展示 Tree 的判断逻辑

The production-style LightGBM classifier contains hundreds of trees, so printing its entire logic would not be readable. We therefore train an additional depth-4 tree on the same training features. This shallow tree is an explanation model, not a replacement for LightGBM.  
原有 LightGBM 分类器包含几百棵树，无法把全部规则清楚地打印出来。因此这里额外训练一棵深度为 4 的浅层决策树。它的主要作用是展示判断逻辑，不是取代 LightGBM。

In [ ]:
# Cell 17 - Cell 4 - Train and display Tree logic / 训练并展示 Tree 判断逻辑


def format_lightgbm_node(node, feature_names, depth=0, max_depth=3):
    """Render readable rules from a LightGBM tree node. / 将 LightGBM 树节点转换成可读规则。"""
    # Print a readable prefix of one real LightGBM tree. / 打印真实 LightGBM 中一棵树的前几层规则。
    indent = "  " * depth
    if "leaf_value" in node:
        return [f"{indent}LEAF raw_score={node['leaf_value']:.5f}"]
    feature = feature_names[int(node["split_feature"])]
    threshold = node.get("threshold")
    lines = [
        f"{indent}IF {feature} {node.get('decision_type', '<=')} {threshold} "
        f"(missing_goes_left={node.get('default_left', False)}):"
    ]
    if depth >= max_depth:
        lines.append(f"{indent}  ... deeper branches omitted / 更深分支省略")
        return lines
    lines.extend(format_lightgbm_node(node["left_child"], feature_names, depth + 1, max_depth))
    lines.append(f"{indent}ELSE:")
    lines.extend(format_lightgbm_node(node["right_child"], feature_names, depth + 1, max_depth))
    return lines


# Show the first actual tree from the trained LightGBM ensemble. / 展示已训练 LightGBM 集成模型中的第一棵真实树。
lightgbm_dump = tree_ensemble.booster_.dump_model()
first_lightgbm_tree = lightgbm_dump["tree_info"][0]["tree_structure"]
first_tree_rules = "\n".join(
    format_lightgbm_node(first_lightgbm_tree, list(final_features), max_depth=3)
)
print("First real LightGBM tree (partial) / 第一棵真实 LightGBM 树（部分规则）:")
print(first_tree_rules)
print("\nNote: the final LightGBM probability combines all trees, not only this one.")
print("注意：最终 LightGBM 概率由全部树共同决定，不是只看这一棵。")

shallow_tree = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=150,
    random_state=RANDOM_STATE,
)
shallow_tree.fit(train_df[final_features], train_df["has_tip"].astype(int))
shallow_tree_prediction = shallow_tree.predict(X_shared).astype(int)

tree_rules_text = export_text(
    shallow_tree,
    feature_names=list(final_features),
    decimals=2,
    max_depth=4,
)
print("Decision rules / 决策规则:")
print(tree_rules_text)

plt.figure(figsize=(24, 10))
plot_tree(
    shallow_tree,
    feature_names=final_features,
    class_names=["no tip", "has tip"],
    filled=True,
    rounded=True,
    proportion=True,
    precision=3,
    fontsize=8,
)
plt.title("Explainable depth-4 tree for recorded-tip yes/no / 是否记录小费的深度 4 决策树")
plt.tight_layout()
plt.show()

(LLM_OUTPUT_DIR / "lightgbm_first_tree_partial_rules.txt").write_text(first_tree_rules, encoding="utf-8")
(LLM_OUTPUT_DIR / "explainable_tree_rules.txt").write_text(tree_rules_text, encoding="utf-8")

## 12.4 Build LLM inputs without leaking the answer / 构造不泄漏答案的大模型输入

The LLM receives pickup/dropoff time and H3, fare, distance, duration, speed, route frequency, weekend and holiday flags. It does **not** receive `tip`, `has_tip`, Tree predictions, or the correct answer.  
大模型会看到上下车时间和 H3、车费、距离、时长、速度、路线频率、周末和节假日标记。它**不会**看到 `tip`、`has_tip`、Tree 预测或正确答案。

In [ ]:
# Cell 18 - Cell 5 - Create compact LLM cases and the prompt / 构造大模型样本与提示词

DAY_NAMES = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]


def make_llm_case(row):
    """Build one answer-free trip summary for LLM classification. / 构造一条不泄漏答案的行程摘要供大模型分类。"""
    # Keep only information available to the Tree; never include tip or has_tip. / 只保留 Tree 可用的信息；绝不加入 tip 或 has_tip。
    return {
        "case_id": row["case_id"],
        "pickup_hour": int(row["pickup_hour"]),
        "pickup_weekday": DAY_NAMES[int(row["pickup_dow"])],
        "pickup_month": int(row["pickup_month"]),
        "pickup_is_weekend": int(row["pickup_is_weekend"]),
        "pickup_is_holiday": int(row["pickup_is_holiday"]),
        "pickup_h3": str(row["pickup_h3"]),
        "dropoff_hour": int(row["dropoff_hour"]),
        "dropoff_weekday": DAY_NAMES[int(row["dropoff_dow"])],
        "dropoff_h3": str(row["dropoff_h3"]),
        "same_h3": int(row["same_h3"]),
        "crosses_date": int(row["crosses_date"]),
        "route_training_frequency": int(row["route_train_frequency"]),
        "fare_usd": round(float(row["fare"]), 2),
        "trip_miles": round(float(row["trip_miles"]), 3),
        "trip_seconds": int(row["trip_seconds"]),
        "duration_minutes": round(float(row["timestamp_duration_minutes"]), 2),
        "speed_mph": round(float(row["speed_mph"]), 2),
    }


llm_cases = [make_llm_case(row) for _, row in llm_eval_df.iterrows()]

SYSTEM_PROMPT = '''You are a binary classifier for Chicago TNP trip records.
For each completed trip, predict whether the dataset records a positive tip.
Answer yes when recorded tip is likely greater than zero; otherwise answer no.
Use only the supplied attributes. Do not assume that unrecorded cash tips are visible.
Return one JSON object with key predictions. predictions must be an array with exactly one item per case.
Each item must contain case_id, has_tip (yes or no), confidence (0 to 1), and a concise Chinese reason under 30 Chinese characters.
Do not include markdown or any text outside the JSON object.'''

print("Example sent to the LLM / 发给大模型的示例:")
print(json.dumps(llm_cases[0], indent=2, ensure_ascii=False))
print("\nConfirmed excluded fields / 已确认排除字段: tip, has_tip, Tree prediction, correct answer")

## 12.5 Call the LLM in small batches / 分批调用大模型

Cell 2 requires a valid API key and verifies model access. Responses are cached as JSONL, so rerunning the notebook does not pay for completed cases again. Direct HTTPS is used instead of `client.responses`, avoiding SDK-version mismatch.  
Cell 2 会要求输入有效 API key，并验证模型访问权限。回答会缓存为 JSONL，重新运行时不会重复支付已完成样本。这里直接使用 HTTPS，而不是 `client.responses`，避免 OpenAI SDK 版本不一致问题。

In [ ]:
# Cell 19 - Cell 6 - Run or resume LLM classification / 执行或继续大模型分类


def load_llm_cache(path):
    """Load completed LLM classifications from JSONL cache. / 从 JSONL 缓存读取已完成的大模型分类。"""
    records = {}
    if path.exists():
        for line in path.read_text(encoding="utf-8").splitlines():
            if line.strip():
                item = json.loads(line)
                records[item["case_id"]] = item
    return records


def parse_json_content(content):
    """Parse and validate JSON returned by the LLM. / 解析并验证大模型返回的 JSON。"""
    content = content.strip()
    if content.startswith("```"):
        content = content.strip("`")
        if content.lower().startswith("json"):
            content = content[4:].strip()
    parsed = json.loads(content)
    predictions = parsed.get("predictions", [])
    if not isinstance(predictions, list):
        raise ValueError("predictions must be a list")
    return predictions


def call_llm_batch(batch):
    """Classify one batch with the selected LLM. / 使用选定大模型分类一批样本。"""
    payload = {
        "model": LLM_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": "Classify these trips:\n" + json.dumps(batch, ensure_ascii=False),
            },
        ],
        "response_format": {"type": "json_object"},
        "reasoning_effort": "none",
        "max_completion_tokens": 1600,
    }
    response = requests.post(
        "https://api.openai.com/v1/chat/completions",
        headers={
            "Authorization": f"Bearer {OPENAI_API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=180,
    )
    if response.status_code >= 400:
        raise RuntimeError(f"OpenAI API {response.status_code}: {response.text[:1000]}")
    body = response.json()
    content = body["choices"][0]["message"]["content"]
    return parse_json_content(content)


llm_cache = load_llm_cache(LLM_CACHE_PATH)
pending = [case for case in llm_cases if case["case_id"] not in llm_cache]
print(f"Cached / 已缓存: {len(llm_cache)}, pending / 待处理: {len(pending)}")

if not OPENAI_API_KEY:
    raise RuntimeError("Run Experiment 2 Cell 2 and enter the API key first. / 请先运行实验二 Cell 2 并输入 API key。")

for start in range(0, len(pending), LLM_BATCH_SIZE):
    batch = pending[start:start + LLM_BATCH_SIZE]
    expected_ids = {item["case_id"] for item in batch}
    last_error = None
    for attempt in range(1, LLM_MAX_RETRIES + 1):
        try:
            predictions = call_llm_batch(batch)
            returned = {}
            for item in predictions:
                case_id = str(item.get("case_id", ""))
                answer = str(item.get("has_tip", "")).strip().lower()
                if case_id in expected_ids and answer in {"yes", "no"}:
                    returned[case_id] = {
                        "case_id": case_id,
                        "has_tip": answer,
                        "confidence": float(np.clip(float(item.get("confidence", 0.5)), 0, 1)),
                        "reason": str(item.get("reason", ""))[:300],
                        "model": LLM_MODEL,
                    }
            missing = expected_ids - set(returned)
            if missing:
                raise ValueError(f"Missing or invalid cases: {sorted(missing)}")
            with LLM_CACHE_PATH.open("a", encoding="utf-8") as handle:
                for case in batch:
                    record = returned[case["case_id"]]
                    handle.write(json.dumps(record, ensure_ascii=False) + "\n")
                    llm_cache[record["case_id"]] = record
            last_error = None
            break
        except Exception as exc:
            last_error = exc
            wait_seconds = 2 ** attempt
            print(f"Attempt {attempt} failed / 第 {attempt} 次失败: {exc}")
            time.sleep(wait_seconds)
    if last_error is not None:
        raise RuntimeError(
            f"LLM batch failed after retries: {[x['case_id'] for x in batch]}. "
            f"Last error: {last_error} / 当前批次重试后仍失败，已停止评估。"
        )
    print(f"Completed / 已完成: {len(llm_cache)}/{len(llm_cases)}")

llm_cache = load_llm_cache(LLM_CACHE_PATH)
print("Valid cached predictions / 有效缓存预测:", len(llm_cache))
expected_case_ids = {case["case_id"] for case in llm_cases}
missing_case_ids = expected_case_ids - set(llm_cache)
if missing_case_ids:
    raise RuntimeError(
        f"LLM predictions are incomplete: {len(missing_case_ids)} missing. "
        "Rerun this cell to resume. / 大模型预测未完成，请重新运行本 Cell 继续。"
    )
print("LLM prediction coverage verified / 大模型预测覆盖验证成功: 100%")

## 12.6 Compare accuracy on common rows / 在共同样本上比较准确率

Accuracy alone can look high when zero-tip trips dominate. Balanced accuracy and F1 are therefore reported too. The comparison also reports LLM coverage; an incomplete API run must not be mistaken for a full evaluation.  
当零小费行程占多数时，只看准确率可能会产生误导，所以同时报告平衡准确率和 F1。这里还会报告大模型覆盖率，API 没跑完整时不能误认为完成了全部评估。

In [ ]:
# Cell 20 - Cell 7 - Evaluate Tree and LLM on identical rows / 在相同行程上评估 Tree 与大模型


def binary_metrics(name, y_true, y_pred, rows, coverage=1.0):
    """Calculate binary classification metrics on common rows. / 在共同样本上计算二分类指标。"""
    return {
        "model": name,
        "rows": int(rows),
        "coverage": float(coverage),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
    }


llm_result_rows = []
for case_id, item in llm_cache.items():
    llm_result_rows.append({
        "case_id": case_id,
        "llm_prediction": 1 if item["has_tip"] == "yes" else 0,
        "llm_confidence": float(item["confidence"]),
        "llm_reason": item["reason"],
    })
llm_result_df = pd.DataFrame(llm_result_rows)

comparison_df = llm_eval_df[["case_id", "has_tip", "tip"]].copy()
comparison_df["majority_prediction"] = majority_prediction
comparison_df["shallow_tree_prediction"] = shallow_tree_prediction
comparison_df["lightgbm_prediction"] = tree_prediction
comparison_df["lightgbm_probability"] = tree_probability
if len(llm_result_df):
    comparison_df = comparison_df.merge(llm_result_df, on="case_id", how="left")
else:
    comparison_df["llm_prediction"] = np.nan
    comparison_df["llm_confidence"] = np.nan
    comparison_df["llm_reason"] = ""

common = comparison_df[comparison_df["llm_prediction"].notna()].copy()
coverage = len(common) / len(comparison_df)
if len(common) != len(comparison_df):
    raise RuntimeError(
        f"LLM coverage is {len(common)}/{len(comparison_df)}, not 100%. "
        "Run Experiment 2 Cell 6 again before evaluation. / 大模型覆盖率不足，请先重新运行实验二 Cell 6。"
    )

metric_rows = [
    binary_metrics("Majority baseline", y_shared, majority_prediction, len(y_shared)),
    binary_metrics("Explainable depth-4 tree", y_shared, shallow_tree_prediction, len(y_shared)),
    binary_metrics("LightGBM tree ensemble", y_shared, tree_prediction, len(y_shared)),
]

y_common = common["has_tip"].astype(int).to_numpy()
metric_rows.append(binary_metrics(
    f"LLM - {LLM_MODEL}",
    y_common,
    common["llm_prediction"].astype(int).to_numpy(),
    len(common),
    coverage,
))

llm_tree_metrics = pd.DataFrame(metric_rows).sort_values("F1", ascending=False)
display(llm_tree_metrics.round(4))
print(f"LLM coverage / 大模型覆盖率: {coverage:.1%} ({len(common)}/{len(comparison_df)})")

plot_df = llm_tree_metrics.sort_values("accuracy")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(plot_df["model"], plot_df["accuracy"], color="#2E74B5")
axes[0].set_xlim(0, 1)
axes[0].set_title("Recorded-tip yes/no accuracy / 是否有小费准确率")
axes[1].barh(plot_df["model"], plot_df["F1"], color="#E67E22")
axes[1].set_xlim(0, 1)
axes[1].set_title("Positive-tip F1 / 正小费 F1")
plt.tight_layout()
plt.show()

models_for_matrix = [
    ("LightGBM Tree", y_shared, tree_prediction),
    ("Explainable Tree", y_shared, shallow_tree_prediction),
]
models_for_matrix.append((
    "LLM",
    common["has_tip"].astype(int).to_numpy(),
    common["llm_prediction"].astype(int).to_numpy(),
))

fig, axes = plt.subplots(1, len(models_for_matrix), figsize=(5 * len(models_for_matrix), 4))
if len(models_for_matrix) == 1:
    axes = [axes]
for ax, (name, actual, predicted) in zip(axes, models_for_matrix):
    ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrix(actual, predicted, labels=[0, 1]),
        display_labels=["no tip", "has tip"],
    ).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.show()

## 12.7 Inspect disagreements and Tree evidence / 检查分歧与 Tree 证据

The most useful cases are not the easy agreements, but the rows where LLM and Tree disagree. For those rows, LightGBM `pred_contrib=True` decomposes the prediction into feature contributions. Positive contributions push toward `has tip`; negative contributions push toward `no tip`.  
最值得看的不是两种方法都答对的简单样本，而是大模型与 Tree 意见不同的行程。对这些样本，LightGBM 的 `pred_contrib=True` 会把判断拆成特征贡献：正贡献推动“有小费”，负贡献推动“无小费”。

In [ ]:
# Cell 21 - Cell 8 - Explain disagreements with local Tree contributions / 用局部贡献解释分歧

if len(common):
    common["tree_llm_disagree"] = (
        common["lightgbm_prediction"].astype(int) != common["llm_prediction"].astype(int)
    )
    common["tree_correct"] = common["lightgbm_prediction"].astype(int) == common["has_tip"].astype(int)
    common["llm_correct"] = common["llm_prediction"].astype(int) == common["has_tip"].astype(int)

    disagreement_ids = common.loc[common["tree_llm_disagree"], "case_id"].head(10).tolist()
    disagreement_rows = llm_eval_df[llm_eval_df["case_id"].isin(disagreement_ids)].copy()

    if len(disagreement_rows):
        contributions = tree_ensemble.predict(
            disagreement_rows[final_features],
            pred_contrib=True,
        )
        explanation_rows = []
        for row_position, (_, row) in enumerate(disagreement_rows.iterrows()):
            feature_contrib = contributions[row_position, :-1]
            top_indices = np.argsort(np.abs(feature_contrib))[::-1][:5]
            for rank, feature_index in enumerate(top_indices, 1):
                explanation_rows.append({
                    "case_id": row["case_id"],
                    "rank": rank,
                    "feature": final_features[feature_index],
                    "feature_value": float(row[final_features[feature_index]]),
                    "tree_contribution": float(feature_contrib[feature_index]),
                })
        tree_local_explanations = pd.DataFrame(explanation_rows)
        disagreement_view = common.loc[common["case_id"].isin(disagreement_ids), [
            "case_id", "has_tip", "lightgbm_prediction", "lightgbm_probability",
            "llm_prediction", "llm_confidence", "llm_reason", "tree_correct", "llm_correct"
        ]]
        print("Tree vs LLM disagreements / Tree 与大模型分歧样本:")
        display(disagreement_view)
        print("Top Tree evidence for each disagreement / 每个分歧样本最重要的 Tree 证据:")
        display(tree_local_explanations)
    else:
        tree_local_explanations = pd.DataFrame()
        print("No disagreement in completed LLM rows. / 已完成样本中没有分歧。")
else:
    tree_local_explanations = pd.DataFrame()
    print("Run the API first to inspect disagreements. / 请先完成 API 调用。")

In [ ]:
# Cell 22 - Cell 9 - Export results and print the conclusion / 导出结果并打印结论

llm_tree_metrics.to_csv(LLM_OUTPUT_DIR / "llm_vs_tree_tip_presence_metrics.csv", index=False)
comparison_df.to_csv(LLM_OUTPUT_DIR / "llm_vs_tree_tip_presence_predictions.csv", index=False)
if len(tree_local_explanations):
    tree_local_explanations.to_csv(
        LLM_OUTPUT_DIR / "tree_local_explanations_for_llm_disagreements.csv",
        index=False,
    )

print("=" * 95)
print("LLM VS TREE: RECORDED TIP YES/NO / 大模型与 TREE：是否记录小费")
print("=" * 95)
display(llm_tree_metrics.round(4))
print()
print("How to interpret / 如何解释:")
print("1. Accuracy measures all yes/no decisions; F1 focuses on the positive-tip class.")
print("   Accuracy 衡量全部判断；F1 更关注正小费类别是否被正确识别。")
print("2. The depth-4 tree shows readable rules; LightGBM is the stronger Tree benchmark.")
print("   深度 4 决策树负责展示规则；LightGBM 是更强的 Tree 对照模型。")
print("3. A fair LLM conclusion requires 100% coverage on the fixed shared sample.")
print("   只有大模型完成固定共同样本的 100% 覆盖，才能得出公平结论。")
print("4. The target is recorded positive tip, not every possible cash tip.")
print("   目标是数据中记录的正小费，不代表全部可能存在的现金小费。")
print("5. The LLM sees compact trip summaries, not the full MatrixOne table.")
print("   大模型读取的是压缩后的行程摘要，不是把 MatrixOne 全表直接喂进去。")
print()
print("Outputs / 输出目录:", LLM_OUTPUT_DIR)

## 12.8 Reporting guide / 汇报要点

1. **Why compare on the same rows?** Different rows can make one model look artificially better. / 不同样本会让某个模型看起来虚假地更好。  
2. **Why show two Tree models?** LightGBM is the accuracy benchmark; the shallow tree exposes readable rules. / LightGBM 用于准确率对照，浅层树用于展示可读规则。  
3. **Why not feed all trips to the LLM?** MatrixOne first reduces the task to representative structured cases; this controls cost and makes evaluation possible. / 先把任务压缩成代表性结构化样本，才能控制费用并进行评价。  
4. **What would count as success?** Compare LLM accuracy/F1 with LightGBM, majority baseline, API cost, latency, and explanation usefulness. / 成功不能只看准确率，还要比较 F1、成本、速度和解释价值。  
5. **What does this not prove?** Good classification does not prove causality, and recorded tips may omit cash tips. / 分类效果好不代表因果关系，而且记录小费可能遗漏现金小费。

## Result snapshot / 结果快照

- Expected-tip MAE improves from $1.576 for the training-mean baseline to about $1.415 for the modeled feature groups. / 记录小费期望值 MAE 从训练均值基线的 1.576 美元降到约 1.415 美元。
- Actual dropoff time and location add little improvement over pickup-time information in this sample. / 在该样本中，真实下车时间和地点相对上车时信息带来的额外提升很小。
- On 200 shared yes/no cases, the LLM finds more positive-tip rows, while the tree models mostly predict the majority no-tip class. Accuracy alone is therefore not enough; balanced accuracy, recall, and F1 are required. / 在 200 条共同二分类样本中，大模型能找出更多正小费订单，而树模型大多预测多数类“无小费”。因此不能只看准确率，还要看平衡准确率、召回率和 F1。

**Next / 下一步:** Notebook 04 uses the same compact-input idea for peak continuation instead of tip classification.